In [ ]:
# Import Required Libraries
from flask import Flask, request, jsonify
import joblib
import pandas as pd

In [ ]:
# Load Pre-trained Model and Encoder
model = joblib.load("house_model.pkl")
encoder = joblib.load("location_encoder.pkl")
print("Model and encoder loaded successfully")

In [ ]:
# Create Flask Application
app = Flask(__name__)

@app.route('/')
def home():
    return "Welcome to Mietech LTD House Price Prediction API"

In [ ]:
# Define Prediction API Endpoint
@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        location = data['location']
        sqft = data['sqft']
        rooms = data['rooms']
        
        # Validate inputs
        if not isinstance(location, str) or location not in encoder.classes_:
            return jsonify({'error': 'Invalid location'}), 400
        if not isinstance(sqft, (int, float)) or sqft <= 0:
            return jsonify({'error': 'Invalid sqft, must be positive number'}), 400
        if not isinstance(rooms, int) or rooms <= 0:
            return jsonify({'error': 'Invalid rooms, must be positive integer'}), 400
        
        # Encode location
        location_encoded = encoder.transform([location])[0]
        
        # Prepare input
        input_data = pd.DataFrame([[location_encoded, sqft, rooms]], columns=['location', 'sqft', 'rooms'])
        
        # Predict
        prediction = model.predict(input_data)[0]
        
        return jsonify({'predicted_price': round(prediction, 2)})
    
    except KeyError as e:
        return jsonify({'error': f'Missing field: {str(e)}'}), 400
    except Exception as e:
        return jsonify({'error': 'Server error', 'details': str(e)}), 500

In [ ]:
# Run the Flask Application
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

In [ ]:
# Test API with Sample Requests
# Note: Install requests if not available: !pip install requests
# Run the Flask app in a separate process or terminal before running this cell.

import requests

url = 'http://localhost:5000/predict'

# Valid request (assuming 'Gasabo' is a valid location from the dataset)
data = {'location': 'Gasabo', 'sqft': 1500, 'rooms': 3}
response = requests.post(url, json=data)
print('Valid request response:', response.json())

# Invalid location
data_invalid = {'location': 'InvalidLocation', 'sqft': 1500, 'rooms': 3}
response = requests.post(url, json=data_invalid)
print('Invalid location response:', response.json())

# Missing field
data_missing = {'location': 'Gasabo', 'sqft': 1500}
response = requests.post(url, json=data_missing)
print('Missing field response:', response.json())

# Invalid sqft
data_invalid_sqft = {'location': 'Gasabo', 'sqft': -100, 'rooms': 3}
response = requests.post(url, json=data_invalid_sqft)
print('Invalid sqft response:', response.json())